# Averaged activation trajectory at a fixed line length

This notebook selects examples whose sampled line-length constraint is exactly `FIXED_LINE_LENGTH`, collects the residual-stream activation immediately before the final attention layer, and averages activations at each absolute token position.

A joint PCA is fit across all collected activation vectors. The averaged trajectory is plotted prominently, with faint individual trajectories overlaid to show how representative the average is. Points are colored by the mean cumulative character count at each token position. A grouped distance analysis also reports within-position activation variability after conditioning on exact character count, random-pair scale baselines, and variance reduction from conditioning.

In [4]:
from pathlib import Path
import json

import numpy as np
import plotly.graph_objects as go
import torch
from sklearn.decomposition import PCA
from IPython.display import display

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = Path("../results/line_breaks_results")
CHECKPOINT_PATH = RUN_DIR / "checkpoints" / "final.pt"
FIXED_LINE_LENGTH = 70  # Change this to compare other constraints.
N_EXAMPLES = 500
COLLECTION_BATCH_SIZE = 512
DATA_SEED = 21_001
SHOW_INDIVIDUAL_TRAJECTORIES = True


In [5]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)

device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()

n_layers = cfg["model"]["n_layers"]
final_layer = n_layers - 1
ACTIVATION_SITE = f"blocks.{final_layer}.hook_resid_pre"
seq_len = cfg["task"]["seq_len"]
min_width, max_width = project_data.get_line_length_range(cfg["task"])
if not min_width <= FIXED_LINE_LENGTH <= max_width:
    raise ValueError(f"FIXED_LINE_LENGTH must be between {min_width} and {max_width}.")
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Fixed line length: {FIXED_LINE_LENGTH}")
print(f"Activation site: {ACTIVATION_SITE}")


Building vocab...
('BOS', '_NEWLINE_')
Loaded checkpoint step 10000 on cpu
Fixed line length: 70
Activation site: blocks.2.hook_resid_pre


## Collect exactly fixed-length examples

In [6]:
def character_counts_for_tokens(token_rows):
    all_counts = []
    for row in token_rows.detach().cpu().tolist():
        running_count = 0
        row_counts = []
        for token_id in row:
            if token_id == vocab.newline_id:
                running_count = 0
            else:
                running_count += project_data.extract_character_count(vocab.decode_token(token_id))
            row_counts.append(running_count)
        all_counts.append(row_counts)
    return torch.tensor(all_counts, dtype=torch.long)

@torch.no_grad()
def collect_fixed_length_examples():
    activation_chunks = []
    token_chunks = []
    count_chunks = []
    collected = 0
    batch_number = 0
    while collected < N_EXAMPLES:
        seed = DATA_SEED + batch_number * COLLECTION_BATCH_SIZE
        batch = project_data.make_batch(
            batch_size=COLLECTION_BATCH_SIZE,
            vocab=vocab,
            task_cfg=cfg["task"],
            device=device,
            seed=seed,
        )
        keep = batch.line_lengths == FIXED_LINE_LENGTH
        n_keep = int(keep.sum().item())
        if n_keep:
            _, cache = model.run_with_cache(
                batch.tokens,
                prepend_bos=False,
                names_filter=lambda name: name == ACTIVATION_SITE,
            )
            keep_cpu = keep.detach().cpu()
            activation_chunks.append(cache[ACTIVATION_SITE].detach().cpu()[keep_cpu])
            token_chunks.append(batch.tokens.detach().cpu()[keep_cpu])
            count_chunks.append(character_counts_for_tokens(batch.tokens)[keep_cpu])
            collected += n_keep
        batch_number += 1

    activations = torch.cat(activation_chunks, dim=0)[:N_EXAMPLES].float().numpy()
    tokens = torch.cat(token_chunks, dim=0)[:N_EXAMPLES].numpy()
    character_counts = torch.cat(count_chunks, dim=0)[:N_EXAMPLES].numpy()
    return activations, tokens, character_counts, batch_number

activations, tokens, character_counts, batches_used = collect_fixed_length_examples()
mean_trajectory = activations.mean(axis=0)
mean_character_counts = character_counts.mean(axis=0)
print(f"Collected {len(activations):,} examples using {batches_used:,} generated batches.")
print(f"Activation tensor: {activations.shape}")
print(f"Observed newline fraction by position: {(tokens == vocab.newline_id).mean():.3f}")


Collected 500 examples using 51 generated batches.
Activation tensor: (500, 150, 128)
Observed newline fraction by position: 0.055


## Joint PCA and averaged trajectory

In [7]:
# Fit PCA once across all examples and positions, then project both the
# individual trajectories and the mean trajectory into that shared basis.
pca = PCA(n_components=3)
pca.fit(activations.reshape(-1, activations.shape[-1]))
all_coordinates = pca.transform(activations.reshape(-1, activations.shape[-1])).reshape(activations.shape[0], seq_len, 3)
mean_coordinates = pca.transform(mean_trajectory)

print(f"Joint PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}")

fig = go.Figure()
if SHOW_INDIVIDUAL_TRAJECTORIES:
    for trajectory in all_coordinates:
        fig.add_trace(go.Scatter3d(
            x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
            mode="lines", line=dict(color="rgba(100,100,100,0.08)", width=1),
            showlegend=False, hoverinfo="skip",
        ))

origin = np.zeros(3)
mean_segments = []
for point in mean_coordinates:
    mean_segments.extend([origin, point, [None, None, None]])
fig.add_trace(go.Scatter3d(
    x=[point[0] for point in mean_segments],
    y=[point[1] for point in mean_segments],
    z=[point[2] for point in mean_segments],
    mode="lines", line=dict(color="rgba(0,0,0,0.22)", width=2),
    name="Mean vectors", hoverinfo="skip",
))
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode="markers", marker=dict(size=8, color="black"),
    name="Origin", hovertemplate="(0, 0, 0)<extra>Origin</extra>",
))
fig.add_trace(go.Scatter3d(
    x=mean_coordinates[:, 0], y=mean_coordinates[:, 1], z=mean_coordinates[:, 2],
    mode="lines+markers",
    marker=dict(size=6, color=mean_character_counts, colorscale="Viridis",
                colorbar=dict(title="Mean character count")),
    line=dict(color="royalblue", width=5), name="Mean trajectory",
    text=[f"position {i}" for i in range(seq_len)],
    customdata=np.column_stack([mean_character_counts, (tokens == vocab.newline_id).mean(axis=0)]),
    hovertemplate=("%{text}<br>mean character count=%{customdata[0]:.2f}<br>"
                   "newline frequency=%{customdata[1]:.2%}<extra>Mean trajectory</extra>"),
))
fig.update_layout(
    title=(f"Mean pre-final-attention activation trajectory at line length {FIXED_LINE_LENGTH}<br>"
           f"{N_EXAMPLES:,} examples; joint PCA variance explained = {pca.explained_variance_ratio_.sum():.1%}"),
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    height=850, margin=dict(l=0, r=0, b=0, t=100),
)
display(fig)


Joint PCA variance explained: 91.5%


## Within-position distances after controlling for character count

For each absolute token position and exact cumulative character count, this section groups examples together and computes the mean pairwise Euclidean distance between their full pre-final-attention activation vectors. Groups with fewer than two examples cannot contribute a pairwise distance.


In [9]:
distance_records = []
for position in range(seq_len):
    counts_at_position = np.unique(character_counts[:, position])
    for character_count in counts_at_position:
        group = activations[character_counts[:, position] == character_count, position]
        n_group = len(group)
        n_pairs = n_group * (n_group - 1) // 2
        if n_pairs == 0:
            mean_distance = np.nan
        else:
            # torch.pdist computes all unique unordered pairs without
            # constructing an explicit n x n distance matrix.
            group_tensor = torch.as_tensor(group, dtype=torch.float32)
            mean_distance = float(torch.pdist(group_tensor, p=2).mean().item())
        distance_records.append({
            "token_position": position,
            "character_count": int(character_count),
            "n_examples": n_group,
            "n_pairs": n_pairs,
            "mean_full_activation_distance": mean_distance,
        })

distance_table = __import__("pandas").DataFrame.from_records(distance_records)
distance_table = distance_table.sort_values(["token_position", "character_count"])
display(distance_table.head(30))
print(
    f"Groups with at least two examples: "
    f"{(distance_table['n_pairs'] > 0).sum():,} / {len(distance_table):,}"
)
print(
    "Mean of grouped mean distances: "
    f"{distance_table['mean_full_activation_distance'].mean():.4f}"
)

# Heatmap: columns are absolute token positions, rows are exact character counts.
heatmap_values = distance_table.pivot(
    index="character_count",
    columns="token_position",
    values="mean_full_activation_distance",
).sort_index()
fig_distances = go.Figure(go.Heatmap(
    z=heatmap_values.to_numpy(),
    x=heatmap_values.columns.to_numpy(),
    y=heatmap_values.index.to_numpy(),
    colorscale="Viridis",
    colorbar=dict(title="Mean full-space distance"),
    hovertemplate=(
        "token position=%{x}<br>character count=%{y}<br>"
        "mean pairwise distance=%{z:.4f}<extra></extra>"
    ),
))
fig_distances.update_layout(
    title=(f"Within-group activation distance at fixed line length {FIXED_LINE_LENGTH}"),
    xaxis_title="Absolute token position",
    yaxis_title="Character count",
    height=700, margin=dict(l=80, r=40, b=70, t=90),
)
display(fig_distances)


,token_position,character_count,n_examples,n_pairs,mean_full_activation_distance
0,0,0,500,124750,0.000000
1,1,1,71,2485,1.161016
2,1,2,58,1653,1.343048
3,1,3,72,2556,1.298838
4,1,4,49,1176,1.371046
5,1,5,55,1485,1.399801
6,1,6,70,2415,1.408145
7,1,7,57,1596,1.266764
8,1,8,68,2278,1.063048
9,2,2,8,28,1.130688


Groups with at least two examples: 9,375 / 10,177
Mean of grouped mean distances: 4.4669


## Random-pair scale and variance reduction

The first random-pair baseline samples two examples at the same token position but ignores character count, making it directly comparable to the conditioned distances above. A second baseline samples activation vectors completely at random across examples and positions.

Variance reduction is measured using the trace of the activation covariance, equivalently the mean squared distance from the relevant group mean.


In [10]:
RANDOM_PAIR_SEED = 21_002
N_RANDOM_PAIRS = 20_000
rng = np.random.default_rng(RANDOM_PAIR_SEED)
n_examples, n_positions, d_model = activations.shape

# Random pairs at the same token position, without character-count conditioning.
random_positions = rng.integers(0, n_positions, size=N_RANDOM_PAIRS)
random_a = rng.integers(0, n_examples, size=N_RANDOM_PAIRS)
random_b = rng.integers(0, n_examples, size=N_RANDOM_PAIRS)
equal = random_a == random_b
while equal.any():
    random_b[equal] = rng.integers(0, n_examples, size=equal.sum())
    equal = random_a == random_b
same_position_distances = np.linalg.norm(
    activations[random_a, random_positions] - activations[random_b, random_positions],
    axis=1,
)

# Completely random pairs across all example/position activations.
flat_activations = activations.reshape(-1, d_model)
flat_size = len(flat_activations)
flat_a = rng.integers(0, flat_size, size=N_RANDOM_PAIRS)
flat_b = rng.integers(0, flat_size, size=N_RANDOM_PAIRS)
equal = flat_a == flat_b
while equal.any():
    flat_b[equal] = rng.integers(0, flat_size, size=equal.sum())
    equal = flat_a == flat_b
all_position_distances = np.linalg.norm(
    flat_activations[flat_a] - flat_activations[flat_b],
    axis=1,
)

# Total variance and residual variance after conditioning.
global_mean = flat_activations.mean(axis=0)
total_variance = float(np.mean((flat_activations - global_mean) ** 2))

position_means = activations.mean(axis=0)
position_residual_variance = float(np.mean((activations - position_means[None, :, :]) ** 2))

within_group_squared_error = 0.0
group_count = 0
for position in range(n_positions):
    for character_count in np.unique(character_counts[:, position]):
        group = activations[character_counts[:, position] == character_count, position]
        if len(group) == 0:
            continue
        group_mean = group.mean(axis=0)
        within_group_squared_error += float(np.sum((group - group_mean) ** 2))
        group_count += len(group)
position_character_residual_variance = within_group_squared_error / (group_count * d_model)

print(
    f"Same-position random-pair distance: {same_position_distances.mean():.4f} "
    f"± {same_position_distances.std():.4f}"
)
print(
    f"Completely random-pair distance: {all_position_distances.mean():.4f} "
    f"± {all_position_distances.std():.4f}"
)
print(f"Total activation variance: {total_variance:.6f}")
print(f"Residual variance after conditioning on token position: {position_residual_variance:.6f}")
print(
    f"Residual variance after conditioning on position + character count: "
    f"{position_character_residual_variance:.6f}"
)
print(
    f"Variance reduction from token position: "
    f"{1 - position_residual_variance / total_variance:.2%}"
)
print(
    f"Additional variance reduction from character count given position: "
    f"{1 - position_character_residual_variance / position_residual_variance:.2%}"
)
print(
    f"Total variance reduction from position + character count: "
    f"{1 - position_character_residual_variance / total_variance:.2%}"
)


Same-position random-pair distance: 20.2218 ± 15.2001
Completely random-pair distance: 29.2995 ± 14.8646
Total activation variance: 4.187078
Residual variance after conditioning on token position: 2.482008
Residual variance after conditioning on position + character count: 0.078987
Variance reduction from token position: 40.72%
Additional variance reduction from character count given position: 96.82%
Total variance reduction from position + character count: 98.11%


## Mean trajectory in 2D PCA

In [8]:
pca_2d = PCA(n_components=2)
pca_2d.fit(activations.reshape(-1, activations.shape[-1]))
mean_coordinates_2d = pca_2d.transform(mean_trajectory)
fig_2d = go.Figure()
fig_2d.add_trace(go.Scatter(
    x=[0], y=[0], mode="markers", marker=dict(size=9, color="black"),
    name="Origin", hovertemplate="(0, 0)<extra>Origin</extra>",
))
fig_2d.add_trace(go.Scatter(
    x=mean_coordinates_2d[:, 0], y=mean_coordinates_2d[:, 1], mode="lines+markers",
    marker=dict(size=7, color=mean_character_counts, colorscale="Viridis",
                colorbar=dict(title="Mean character count")),
    line=dict(color="royalblue", width=4), name="Mean trajectory",
    text=[f"position {i}" for i in range(seq_len)],
    customdata=np.column_stack([mean_character_counts, (tokens == vocab.newline_id).mean(axis=0)]),
    hovertemplate=("%{text}<br>mean character count=%{customdata[0]:.2f}<br>"
                   "newline frequency=%{customdata[1]:.2%}<extra>Mean trajectory</extra>"),
))
fig_2d.update_layout(
    title=(f"Mean pre-final-attention trajectory at line length {FIXED_LINE_LENGTH} — 2D PCA<br>"
           f"Variance explained = {pca_2d.explained_variance_ratio_.sum():.1%}"),
    xaxis_title="PC1", yaxis_title="PC2", height=700,
    margin=dict(l=60, r=40, b=60, t=100),
)
display(fig_2d)
